# NYC Taxi Trip Duration: Exploratory Data Analysis (EDA)

## Project Overview
This project aims to predict the total ride duration of taxi trips in New York City.

Following the project guidelines, this notebook is dedicated entirely to **Exploratory Data Analysis (EDA) and Feature Engineering (no modeling)**.

Our objective here is to elegantly explore the dataset, formulate our logic, and build mathematical features that will help our algorithms understand the physical reality of NYC taxi routing. All subsequent model training will be handled in separate Python scripts[cite: 8].

## Table of Contents
1. Data Loading & Diagnostics
2. Initial Data Exploration
3. Feature Engineering
    - Datetime Extraction
    - Geographic Distance (Haversine & Manhattan)
    - Travel Direction (Bearing)
    - Airport Proximity
    - Rush-Hour Distance Interactions
    - Spatial Grids and Time-Slot Categories
    - Cyclical Time Features
    - Distance Relationship Features
    - Target Log Transformation
    - Log-Scaled Geographic Distances

### Libraries and Configuration
To keep this notebook elegant and professional, all data processing and feature engineering logic has been abstracted into custom Python modules (`data_helper.py` and `feature_engineering.py`).

In [46]:
import pandas as pd

pd.options.display.max_columns = 100
from taxi_trip_duration.manipulation.data_helper import *
from taxi_trip_duration.manipulation.feature_engineering import *

### Loading the Dataset
We combine the `train.csv` and `val.csv` splits into a single dataframe for comprehensive Exploratory Data Analysis.

In [47]:
train, val, _ = load_dataset()
df = pd.concat([train, val], ignore_index=True)
df.sample(5)

Loading datasets from: dataset/split/


,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
444462,id1850730,1,2016-03-23 15:22:05,1,-73.988243,40.754002,-73.991119,40.750736,N,47
61990,id2178423,1,2016-02-19 13:10:16,1,-73.993752,40.749798,-73.973503,40.757713,N,953
34902,id0797096,1,2016-06-20 07:50:23,1,-73.951332,40.770088,-73.989670,40.736919,N,663
240032,id1526353,2,2016-05-29 00:20:02,2,-73.967163,40.793350,-73.977997,40.783039,N,214
1066345,id3537806,2,2016-04-09 23:47:22,2,-73.979271,40.787159,-73.989357,40.773220,N,278


### Initial Data Exploration: Original Features

Before creating derived variables, review what each original column represents and how it may relate to trip duration.

- **`id`** — identifier for a trip record.
- **`vendor_id`** — code identifying the trip-record provider.
- **`pickup_datetime`** — timestamp when the meter was engaged.
- **`passenger_count`** — passenger count recorded by the driver.
- **`pickup_longitude`, `pickup_latitude`** — pickup coordinates.
- **`dropoff_longitude`, `dropoff_latitude`** — drop-off coordinates.
- **`store_and_fwd_flag`** — indicates whether the trip record was stored in vehicle memory before being sent to the provider (`Y`) or sent directly (`N`).
- **`trip_duration`** — trip duration in seconds; this is the prediction target.

This inventory provides the baseline for checking data types, completeness, and the information available for feature engineering.

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1229319 entries, 0 to 1229318
Data columns (total 10 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   id                  1229319 non-null  str    
 1   vendor_id           1229319 non-null  int64  
 2   pickup_datetime     1229319 non-null  str    
 3   passenger_count     1229319 non-null  int64  
 4   pickup_longitude    1229319 non-null  float64
 5   pickup_latitude     1229319 non-null  float64
 6   dropoff_longitude   1229319 non-null  float64
 7   dropoff_latitude    1229319 non-null  float64
 8   store_and_fwd_flag  1229319 non-null  str    
 9   trip_duration       1229319 non-null  int64  
dtypes: float64(4), int64(3), str(3)
memory usage: 93.8 MB


In [49]:
checking_data(df)

Data Diagnostics:
 - Shape              : (1229319, 10)
 - Missing Values     : 0
 - Duplicate Records  : 0


### Data Diagnostics and Quality Assessment

The initial checks on the combined training and validation data indicate the following:

- **Dataset size:** 1,229,319 records and 10 original features.
- **Completeness:** no missing values were found.
- **Duplicate records:** no duplicate rows were found by the check used in this notebook.
- **Data types:** geographic coordinates are stored as `float64`; `passenger_count`, `vendor_id`, and `trip_duration` are stored as `int64`.
- **Timestamp representation:** `pickup_datetime` is currently an object/string column, so it needs parsing before useful calendar and clock components can be extracted.

These checks suggest that explicit missing-value imputation and duplicate-row removal are not required for the inspected data. They do not, by themselves, confirm that all values are plausible: coordinate ranges, unusually short or long trips, and other outliers may still warrant separate investigation.

## Feature Engineering

The following steps derive additional signals from the original trip records. Each feature is motivated by a possible relationship between location, time, and travel duration.

The engineered variables are intended to provide useful representations for later experiments; their predictive value should ultimately be checked using validation data.

In [50]:
df = date_time(df, eda=True)

### Datetime Feature Extraction

A raw timestamp is difficult for many estimators to use directly. We parse `pickup_datetime` and extract calendar and clock components that can represent recurring travel patterns:

- **`pickup_hour`** — hour of the day, capturing daily traffic cycles.
- **`pickup_day_of_week`** — day of the week, capturing weekday–weekend differences.
- **`pickup_month`** — month of the year, allowing for broad seasonal patterns.
- **`pickup_day_of_year`** — position within the year, providing a more detailed calendar index.

Because `eda=True` is supplied, the function also retains `pickup_date` and `pickup_time` for human-readable exploration and plotting. These are exploratory conveniences; downstream modeling should use only features supported by the intended pipeline.

In [51]:
df = haversine_distance(df)

### Haversine Distance (Great-Circle Distance)
A fundamental predictor of travel time is the total distance traveled. Since we are given geographical coordinates in degrees, we calculate the spherical distance between the pickup and dropoff points. 

We use the **Haversine formula**, which calculates the shortest path over the earth's surface using an average earth radius of 6371.0 km.
        
Given latitudes $\phi_1, \phi_2$ and longitudes $\lambda_1, \lambda_2$ in radians, the distance $d$ is calculated as:

$$a = \sin^2\left(\frac{\phi_2 - \phi_1}{2}\right) + \cos(\phi_1) \cdot \cos(\phi_2) \cdot \sin^2\left(\frac{\lambda_2 - \lambda_1}{2}\right)$$
$$c = 2 \cdot \arcsin(\sqrt{a})$$
$$d = 6371.0 \cdot c$$

In [52]:
df = manhattan_distance(df)

### Manhattan Distance (L1 Norm)
While Haversine provides the absolute shortest distance, cars in New York City navigate a strict grid system of avenues and streets. 

The **Manhattan-style distance** approximates this grid navigation by calculating the sum of the North-South and East-West distances at the mean latitude.

Where $R$ is the Earth's radius (6371.0 km) and $\phi_{mean}$ is the average latitude:

$$d_{NS} = R \cdot |\phi_2 - \phi_1|$$
$$d_{EW} = R \cdot \cos(\phi_{mean}) \cdot |\lambda_2 - \lambda_1|$$
$$d_{Manhattan} = d_{NS} + d_{EW}$$

In [53]:
df = pickup_bearing(df)

### Pickup Bearing (Direction of Travel)
The direction a taxi is heading can heavily influence traffic density and trip duration. We calculate the initial **Bearing** (compass direction) from the pickup point to the dropoff point. 
        
The resulting angles range from -180 to 180 degrees.

Where $\Delta\lambda$ is the difference in longitude:

$$Y = \sin(\Delta\lambda) \cdot \cos(\phi_2)$$
$$X = \cos(\phi_1) \cdot \sin(\phi_2) - \sin(\phi_1) \cdot \cos(\phi_2) \cdot \cos(\Delta\lambda)$$
$$\theta = \arctan2(Y, X) \cdot \frac{180}{\pi}$$

In [ ]:
df = airport_distances(df)

### Airport Distances — Proximity to Major Airports

Airport trips may differ from other journeys because of longer access roads, highway travel, tolls, and airport-specific traffic. To represent proximity to major NYC-area airports, this step calculates geographic distances between trip endpoints and fixed airport reference coordinates.

The airport reference points used are:

- **John F. Kennedy International Airport (JFK):** `40.6413, -73.7781`
- **LaGuardia Airport (LGA):** `40.7769, -73.8740`
- **Newark Liberty International Airport (EWR):** `40.6895, -74.1745`

These distances can help distinguish trips that begin or end near an airport. They measure distance to the selected reference coordinates, not distance to an airport entrance or terminal road. The exact output column names are defined by `airport_distances()`.

In [ ]:
df = time_interactions(df)

### Time Interactions — Distance During Rush Hour

The relationship between distance and duration can vary by time of day. A journey of a given distance during a busy weekday period may take longer than the same journey during a quieter period.

This step creates a rush-hour indicator, `is_rush_hour`, and interactions between that indicator and the two geographic distance estimates:

- **Rush-hour Manhattan distance:** Manhattan distance multiplied by the rush-hour indicator.
- **Rush-hour Haversine distance:** Haversine distance multiplied by the rush-hour indicator.

For a binary indicator, these interaction features are zero outside the defined rush-hour window and equal to the corresponding distance inside it. They allow a model to represent a different distance–duration relationship during the selected period.

The exact rush-hour hours and weekday rules are determined by `time_interactions()`; those rules should be documented there so the definition is explicit and reproducible.

In [ ]:
df = heavy_hitter_categories(df)

### High-Cardinality Categories — Spatial Grids and Time Slots

Continuous coordinates and timestamps can also be represented as categorical groups. This step creates localized categories that may help a model capture patterns associated with particular areas or recurring time slots.

1. **Spatial grids (`pickup_grid` and `dropoff_grid`)** — coordinates are rounded to two decimal places to group nearby locations into the same approximate spatial cell. The physical size of a cell varies with latitude, so the grid should not be interpreted as an exact 1 km × 1 km partition.
2. **Day–hour category (`day_hour`)** — combines the day-of-week and pickup-hour values into a category such as `0_8`. This can represent recurring weekly traffic patterns for a particular hour.

These are engineered categories, not guaranteed indicators of a causal effect. Their usefulness depends on the estimator and encoding strategy used later; high-cardinality categories may also increase the number of model parameters.

In [ ]:
df = cyclical_time(df)

### Cyclical Time Features

Time-related features such as pickup hour and day of the week have a cyclical nature. For example, hour 23 is closer to hour 0 than to hour 12, even though their numerical values suggest otherwise.

Cyclical encoding represents these features using sine and cosine transformations, allowing models to capture their periodic relationships.

For the pickup hour \(h\), the cyclical transformations are:

$$
\text{pickup\_hour\_sin} =
\sin\left(\frac{2\pi h}{24}\right)
$$

$$
\text{pickup\_hour\_cos} =
\cos\left(\frac{2\pi h}{24}\right)
$$

Similarly, for the day of the week \(d\), where Monday is represented by 0
and Sunday by 6:

$$
\text{pickup\_day\_sin} =
\sin\left(\frac{2\pi d}{7}\right)
$$

$$
\text{pickup\_day\_cos} =
\cos\left(\frac{2\pi d}{7}\right)
$$

This representation places the beginning and end of the daily cycle close together. The same principle can be applied to other periodic time features, using the appropriate cycle length.

The resulting features supplement the original time components and allow subsequent experiments to evaluate whether cyclical encoding improves prediction performance.

In [ ]:
df = distance_relationships(df)

### Distance Relationship Features

Haversine and Manhattan distances represent different estimates of the geographic separation between pickup and drop-off locations. Haversine measures great-circle distance, while Manhattan distance approximates travel distance along a grid.

This step derives additional features from the relationship between these distance estimates. These features may help represent differences between direct geographic separation and the grid-based approximation.

The resulting variables supplement the original distance features. Their usefulness should be evaluated during model training to determine whether they provide additional predictive information.

The distance relationship features are calculated as follows:

$$
\text{distance\_ratio} =
\frac{d_{\text{Manhattan}}}
{d_{\text{Haversine}} + \epsilon}
$$

$$
\text{distance\_difference} =
d_{\text{Manhattan}} - d_{\text{Haversine}}
$$

where $\epsilon = 10^{-6}$ is added to the denominator to avoid division by zero.

In [ ]:
df = target_log(df,drop_original=False)

### Target Log Transformation

Trip duration is often right-skewed, meaning that a small number of trips may have considerably longer durations than most trips. Applying a logarithmic transformation can reduce this skewness and make the target distribution easier to model.

The transformation used here is:

$$
y_{\text{log}} = \ln(y + 1)
$$

In the implementation, the transformed target is stored in `log1p_trip_duration`, derived from `trip_duration`. The addition of 1 ensures that the transformation is defined when the duration is zero.

The function is called with `drop_original=False`, so the original `trip_duration` column is retained for EDA while the transformed target is added. During model training, the target must be separated from the predictor features to prevent target leakage.

In [ ]:
df = log_distances(df)

### Log-Scaled Geographic Distances

The derived distance features can also be right-skewed, with a small number of trips having much larger endpoint distances than most trips. Applying `log1p` provides a compressed scale that may be useful for models sensitive to feature scale or linear relationships.

The transformations are:

$$
\text{log\_haversine} = \ln(1 + d_{\text{Haversine}})
$$

$$
\text{log\_manhattan} = \ln(1 + d_{\text{Manhattan}})
$$

The `+1` keeps the transformation defined for a distance of zero. These columns supplement the original distance features rather than replacing them. Whether the transformed versions improve performance should be assessed in the model-validation stage.